In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
import pickle

In [ ]:
BASE = "/content/drive/MyDrive/nuralproject"


CSV_PATH = f"{BASE}/dataset_classification.csv"
IMG_FEAT_PATH = f"{BASE}/resnet_features.npy"
BASE = "/content/drive/MyDrive/nuralproject"



In [ ]:
df = pd.read_csv(CSV_PATH)

df["question"] = df["question"].str.lower().str.strip()
df["answer"] = df["answer"].str.lower().str.strip()

img_features = np.load(IMG_FEAT_PATH)

# align sizes
img_features = img_features[:len(df)]

In [ ]:
answers = df["answer"].unique()

#labelEncoding
ans2idx = {a: i for i, a in enumerate(answers)}
idx2ans = {i: a for a, i in ans2idx.items()}

y = torch.tensor(df["answer"].map(ans2idx).values) #tensor

num_classes = len(ans2idx)
print("Classes:", num_classes)

Classes: 13


In [ ]:
from transformers import BertTokenizer, BertModel

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
bert.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
def get_bert(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=30
    )

    with torch.no_grad():
        out = bert(**inputs) #ON

    return out.pooler_output.squeeze(0) #cls embedding

In [ ]:
bert_features = []

for q in df["question"]:
    bert_features.append(get_bert(q).numpy()) #embeddings

bert_features = np.array(bert_features)

print("BERT shape:", bert_features.shape)

BERT shape: (195, 768)


In [ ]:
import torch

X_img = torch.tensor(img_features).float()
X_txt = torch.tensor(bert_features).float()
y = torch.tensor(y)

Xi_train, Xi_test, Xt_train, Xt_test, y_train, y_test = train_test_split(
    X_img, X_txt, y,
    test_size=0.2,
    random_state=42
)

/tmp/ipykernel_1219/1508263677.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y)


In [ ]:
from torch.utils.data import Dataset, DataLoader

class FusionDataset(Dataset):
    def __init__(self, img, txt, y):
        self.img = img
        self.txt = txt
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx): #img,txt,label
        return self.img[idx], self.txt[idx], self.y[idx]

In [ ]:
train_loader = DataLoader(
    FusionDataset(Xi_train, Xt_train, y_train),
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    FusionDataset(Xi_test, Xt_test, y_test),
    batch_size=32
)

In [ ]:
import torch.nn as nn

class FusionModel(nn.Module):

    def __init__(self, img_dim, txt_dim, num_classes):

        super().__init__()

        self.img_fc = nn.Linear(img_dim, 512)
        self.txt_fc = nn.Linear(txt_dim, 512)

        self.classifier = nn.Sequential(
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, img, txt):

        img = self.img_fc(img)
        txt = self.txt_fc(txt)
#fusion done(img,txt)
        x = torch.cat([img, txt], dim=1)

        return self.classifier(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FusionModel(
    img_dim=X_img.shape[1],
    txt_dim=X_txt.shape[1],
    num_classes=num_classes
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
criterion = nn.CrossEntropyLoss()

best_acc = 0
patience = 10
counter = 0

In [ ]:
for epoch in range(30):

    model.train()
    total_loss = 0

    for img, txt, labels in train_loader:

        img, txt, labels = img.to(device), txt.to(device), labels.to(device)

        optimizer.zero_grad()

        out = model(img, txt)

        loss = criterion(out, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # evaluation
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for img, txt, labels in test_loader:

            img, txt, labels = img.to(device), txt.to(device), labels.to(device)

            out = model(img, txt)
            preds = torch.argmax(out, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total

    print(f"Epoch {epoch} | Loss {avg_loss:.4f} | Acc {acc:.4f}")

    # save best
    if acc > best_acc:
        best_acc = acc
        counter = 0

        torch.save(model.state_dict(),
                   f"{BASE}/models/bert_resnet/fusion_best.pth")

        print("Saved best model")

    else:
        counter += 1
        if counter >= patience:
            print("Early stopping")
            break

Epoch 0 | Loss 2.5447 | Acc 0.1282
Saved best model
Epoch 1 | Loss 2.2766 | Acc 0.1795
Saved best model
Epoch 2 | Loss 1.9237 | Acc 0.2308
Saved best model
Epoch 3 | Loss 1.5392 | Acc 0.2564
Saved best model
Epoch 4 | Loss 1.2606 | Acc 0.5128
Saved best model
Epoch 5 | Loss 0.9759 | Acc 0.5385
Saved best model
Epoch 6 | Loss 0.8505 | Acc 0.5128
Epoch 7 | Loss 0.7175 | Acc 0.6410
Saved best model
Epoch 8 | Loss 0.6136 | Acc 0.6667
Saved best model
Epoch 9 | Loss 0.5312 | Acc 0.6410
Epoch 10 | Loss 0.4884 | Acc 0.6667
Epoch 11 | Loss 0.4358 | Acc 0.7179
Saved best model
Epoch 12 | Loss 0.4153 | Acc 0.6154
Epoch 13 | Loss 0.3384 | Acc 0.7179
Epoch 14 | Loss 0.3453 | Acc 0.5897
Epoch 15 | Loss 0.2829 | Acc 0.7179
Epoch 16 | Loss 0.2447 | Acc 0.7179
Epoch 17 | Loss 0.1895 | Acc 0.6410
Epoch 18 | Loss 0.1875 | Acc 0.7436
Saved best model
Epoch 19 | Loss 0.1523 | Acc 0.7179
Epoch 20 | Loss 0.1714 | Acc 0.6667
Epoch 21 | Loss 0.1338 | Acc 0.7436
Epoch 22 | Loss 0.1499 | Acc 0.7436
Epoch 23 | L

In [ ]:
with open(f"{BASE}/models/bert_resnet/labels.pkl", "wb") as f:
    pickle.dump(ans2idx, f)

np.save(f"{BASE}/models/bert_resnet/bert_features.npy", bert_features)

In [ ]:
model.eval()

for i in range(100):

    img = X_img[i].unsqueeze(0).to(device)
    txt = X_txt[i].unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(img, txt)
        pred = torch.argmax(out, dim=1).item()

    print("REAL:", df.iloc[i]["answer"])
    print("PRED:", idx2ans[pred])
    print("="*40)

REAL: dog
PRED: dog
REAL: dog
PRED: dog
REAL: dog
PRED: dog
REAL: dog
PRED: dog
REAL: dog
PRED: dog
REAL: dog
PRED: dog
REAL: dog
PRED: dog
REAL: dog
PRED: dog
REAL: dog
PRED: dog
REAL: dog
PRED: dog
REAL: dog
PRED: dog
REAL: dog
PRED: dog
REAL: dog
PRED: dog
REAL: dog
PRED: dog
REAL: dog
PRED: dog
REAL: dog
PRED: cat
REAL: cat
PRED: cat
REAL: cat
PRED: cat
REAL: cat
PRED: cat
REAL: cat
PRED: cat
REAL: cat
PRED: cat
REAL: cat
PRED: cat
REAL: cat
PRED: cat
REAL: cat
PRED: cat
REAL: cat
PRED: cat
REAL: cat
PRED: cat
REAL: cat
PRED: cat
REAL: cat
PRED: cat
REAL: cat
PRED: cat
REAL: horse
PRED: cat
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PRED: horse
REAL: horse
PR